# 04 — Counterfactual: what if Brooklyn Pantry & Coffee closed?

Because the LLM scores all 30 candidates per visit (not just the truth), we
can run *counterfactual closures*: drop a POI from every candidate set,
re-normalise the model's top-5 weights, and measure where the lost mass
redistributes.

This notebook reproduces the closure analysis for **Brooklyn Pantry &
Coffee Shop** that anchors the bottom-right of the capstone poster.


## 1. The headline numbers

In [ ]:
import json
c = json.load(open('../results/counterfactual_brooklyn_pantry.json'))
print(f"POI: {c['poi']['name']}")
print(f"Original predicted share: {c['headline_numbers']['original_predicted_share_percent']}%")
print(f"Appears in {c['headline_numbers']['appeared_in_predictions']:,}/{c['headline_numbers']['of_total_predictions']:,}"
      f" predictions ({c['headline_numbers']['presence_rate_percent']}%)")
print(f"Top 15 substitutes capture: {c['headline_numbers']['top_15_substitutes_combined_capture_percent']}% of lost mass")
print(f"Weighted-avg substitute distance: {c['headline_numbers']['weighted_avg_substitute_distance_m']} m")

## 2. Top substitutes

In [ ]:
import pandas as pd
subs = pd.DataFrame(c['top_substitutes'])
subs

## 3. Per-cluster reliance — who feels the loss most?

In [ ]:
rel = pd.DataFrame(c['per_cluster_reliance']).sort_values('reliance_pct', ascending=False)
rel

**Read as**: White Professional Commuters (C5, 10.7% reliance) and Black
Middle-Income workers (C3, 9.1%) lean on Brooklyn Pantry the hardest.
Low-Income Service Workers (C4, 1.6%) barely use it — they substitute to
McDonald's. The LLM's per-cluster persona conditioning makes these
heterogeneous redistributions visible without any explicit demographic
target in the loss.

## 4. The closure substitution map

In [ ]:
from IPython.display import Image
Image('../figures/counterfactual/brooklyn_pantry/fig_closure_Brooklyn_Pantry_and_Coffee_Shop.png')

## 5. Reproduce on a different POI

The same machinery works on any POI in the cluster pool. The script
`scripts/counterfactual_closure.py` (TODO in the public repo) takes a POI
name + the prediction JSONL and writes the substitution map.

Pseudocode:

```python
predictions = load_jsonl('preds_oos_2022.jsonl')
target_poi  = 'Some Restaurant Name|lat|lon'
new_preds   = []
for p in predictions:
    if target_poi not in {c['poi'] for c in p['top_5']}:
        new_preds.append(p)  # closure didn't affect this prediction
        continue
    remaining = [c for c in p['top_5'] if c['poi'] != target_poi]
    z = sum(c['prob'] for c in remaining)
    for c in remaining:
        c['prob'] /= z
    new_preds.append({**p, 'top_5': remaining})
substitution_map = aggregate_by_poi(new_preds) - aggregate_by_poi(predictions)
```
